# 10 · One point becomes a lens

The finite Hermitian curve below has 28 points. Choosing one of them separates
the other 27 into nine triples. Choosing a point outside the curve, and adding
its polar line, instead organizes all 28 into seven quadruples.

\[
28=1+9\cdot3=7\cdot4.
\]

We will **derive** those groups from incidence, verify unique coverage, and use
their measured assignments and ranks to move the same point occurrences.
Along the way, 728 nonzero vectors become 91 projective classes, with eight
representatives per class. Geometric coincidence and quotient formation remain
separate operations.

This extends the conjugation and isotropic-vector experiments in
[Finite-Hermitian-Geometry](https://github.com/virgil-barnard/Finite-Hermitian-Geometry)
into projective incidence and combinatorial designs. Start with lesson 09 if
finite-field coefficient codes are new to you. The larger projective terminology
is introduced alongside each construction.

In [ ]:
from pathlib import Path
from dataclasses import replace
import json
import sys
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import Code, Video, display
from kaleion import Motion, Workspace, param, vector
from kaleion.viewers.plotly import snapshot_figure
from kaleion.viewers.video import write_mp4

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'pyproject.toml').exists() and (p / 'src/kaleion').is_dir())
sys.path.insert(0, str(ROOT / 'notebooks'))
from lesson_views import COLORS, style, profiles, replay, save_figures
pio.renderers.default = 'plotly_mimetype+notebook'

OUTPUT = ROOT / 'build/notebooks/hermitian-partitions'
OUTPUT.mkdir(parents=True,exist_ok=True)
P = 3               # This complete finite example uses F_9 and 91 projective points.
FOCUS = 4           # A curve point: [1:0:1+i]. Try 85 for a point at infinity.
EXTERNAL = 0        # Outside the curve: [1:0:0]. Try 90 for [0:0:1].
GROUP_COLORS = COLORS + ('#e89665','#8ed4cf','#d299d4')

In [ ]:
from functools import reduce
from math import gcd, pi
from kaleion import Collection, F, choose, cos, sin
from quadratic_coordinates import (field_multiply, field_sum, field_conjugate,
                                   field_norm, hermitian_pair, element_label)

## 1 · Eight vector representatives for each projective point

Work over $K=\mathbb F_3[i]$, containing nine elements. A projective point is an
equivalence class of nonzero triples:

\[
(u,v,w)\sim(\lambda u,\lambda v,\lambda w),\qquad \lambda\in K^\times.
\]

Normalize the first nonzero coordinate to 1. Every class then has exactly one
representative of the form $(1,a,b)$, $(0,1,a)$, or $(0,0,1)$.
There are $9^2+9+1=91$ possibilities.

The next construction starts with all $9^3-1=728$ nonzero triples, performs the
normalization symbolically, and **counts** the resulting classes. The field inverse
is $z^{-1}=\overline z\,N(z)^{p-2}$. We select away the zero vector before using it.

In [ ]:
def projective_quotient(p=3):
    if p != 3:
        raise ValueError("The full projective quotient uses the small field F_9")
    q = p*p
    raw = Collection.grid(q,q,q,values=F.i*q*q+F.j*q+F.k).where(F.value != 0).select()
    raw = raw.annotate(u=F.i,v=F.j,w=F.k).arrange(F.i,F.j,F.k)
    leading = raw.annotate(leading=choose(F.u != 0,F.u,choose(F.v != 0,F.v,F.w)))
    inverse = field_multiply(field_conjugate(F.leading,p),field_norm(F.leading,p)**(p-2),p)
    normalized = leading.annotate(U=field_multiply(F.u,inverse,p),
                                  V=field_multiply(F.v,inverse,p),
                                  W=field_multiply(F.w,inverse,p))
    normalized = normalized.with_values(choose(F.U == 1,F.V*q+F.W,
                                                choose(F.V == 1,q*q+F.W,q*q+q)))
    normalized = normalized.arrange(F.U,F.V,F.W)
    class_sizes = normalized.count(by=F.value).order_by(F.key)
    return raw, normalized, class_sizes

def projective_points(p=3, *, labels=None):
    if p != 3:
        raise ValueError("This fully enumerated lesson uses p=3 (91 projective points)")
    q = p*p
    points = Collection.sequence(q*q+q+1, start=0) if labels is None else labels
    points = points.annotate(
        u=choose(F.value < q*q, 1, 0),
        v=choose(F.value < q*q, F.value//q, choose(F.value < q*q+q, 1, 0)),
        w=choose(F.value < q*q, F.value%q, choose(F.value < q*q+q, F.value-q*q, 1)))
    points = points.annotate(self_pair=(field_norm(F.u,p)+field_norm(F.v,p)+field_norm(F.w,p))%p)
    return points.arrange(choose(F.value < q*q, F.v, F.value-q*q),
                          choose(F.value < q*q, F.w, -2))

In [ ]:
raw,normalized,class_sizes = projective_quotient(P)
class_labels = class_sizes.with_values(F.key)
points = projective_points(P,labels=class_labels)
quotient_workspace = Workspace({'raw':raw,'normalized':normalized,
                                'class_sizes':class_sizes,'points':points})
assert not quotient_workspace.state.errors
qr = quotient_workspace.state.results
assert len(qr['raw']) == len(qr['normalized']) == 728
assert qr['class_sizes'].fields['key'].tolist() == list(range(91))
assert qr['class_sizes'].values.tolist() == [8]*91
assert len(qr['points']) == 91
assert qr['raw'].ids == qr['normalized'].ids
assert len(set(map(tuple,qr['normalized'].positions))) == 91

quotient_motion = Workspace({'vectors':raw})
collapse = quotient_motion.set('vectors',normalized,motion=Motion())
expand = quotient_motion.undo()
quotient_samples,quotient_labels = [],[]
for name,transition in [('normalize vector representatives',collapse),('restore representatives',expand)]:
    for fraction in np.linspace(0,1,25):
        quotient_samples.append(transition.frame('vectors',float(fraction)))
        quotient_labels.append(f'{name} · {fraction:.0%}')
quotient_plot = replay(quotient_samples,quotient_labels,
    title='728 occurrences · 91 shared positions · count to form the quotient')
quotient_plot.data[0].marker.size = 4
for frame in quotient_plot.frames: frame.data[0].marker.size = 4
quotient_plot.show()
print('Class counts:',len(qr['class_sizes']),'classes ×',set(qr['class_sizes'].values),'representatives')

At the collapsed endpoint there are still **728 occurrences**, sharing 91
positions. The reduction creates 91 new class occurrences with contributor
provenance. We retain `class_sizes` as the measurement, then relabel its output by
the class key to construct `points`. Those new labels are no longer counts.

## 2 · An isotropic point is a nonzero vector with zero self-pairing

\[
h(x,y)=x_0\overline{y_0}+x_1\overline{y_1}+x_2\overline{y_2},\qquad
\mathcal H=\{[x]:h(x,x)=0\}.
\]

The predicate is well-defined on projective classes because scaling $x$ multiplies
$h(x,x)$ by the nonzero factor $N(\lambda)$. At $p=3$, the point $[1:0:1+i]$
is isotropic: its coordinate norms add to $1+0+2=0$ in the field.

Nondegeneracy of a Hermitian form does not imply positive definiteness. This
continues the counterexample in the Hermitian exposition; it also supplies the
points of a finite geometric object.

Our coordinate chart displays coefficient **codes**. Finite projective lines are
relations on these points, not necessarily straight lines on this drawing.

In [ ]:
curve = points.where(F.self_pair == 0)
curve_snapshot = curve.evaluate()
assert curve_snapshot.cardinality == 28
assert len(curve_snapshot.source) == 91
point_snapshot = curve_snapshot.source
assert int(point_snapshot.fields['self_pair'][FOCUS]) == 0, 'Choose a point on the curve'
assert int(point_snapshot.fields['self_pair'][EXTERNAL]) != 0, 'Choose an external point'
point_labels = [f"[{element_label(u,P)} : {element_label(v,P)} : {element_label(w,P)}]"
                for u,v,w in zip(point_snapshot.fields['u'],point_snapshot.fields['v'],point_snapshot.fields['w'])]
curve_plot = style(go.Figure(go.Scatter(
    x=point_snapshot.positions[:,0],y=point_snapshot.positions[:,1],mode='markers+text',
    text=[str(z) if hit else '' for z,hit in zip(point_snapshot.values,curve_snapshot.mask)],
    textposition='top center',textfont=dict(size=11),
    marker=dict(size=13,color=[COLORS[0] if hit else '#2a3a50' for hit in curve_snapshot.mask]),
    hovertext=point_labels,hovertemplate='%{hovertext}<extra></extra>')),
    '28 Hermitian points among 91 projective classes',height=590)
curve_plot.update_xaxes(title='second coordinate code; bottom strip = infinity',dtick=1)
curve_plot.update_yaxes(title='third coordinate code',dtick=1,scaleanchor='x')
curve_plot.show()

## 3 · Use the same keys for two different roles: points and polar lines

Every projective point $L$ determines a polar line
$L^\perp=\{P:h(L,P)=0\}$. We use the same set of coordinate triples to label these
lines, while keeping the roles explicit: **i is a line pole; j addresses a curve
point, whose projective key is stored as `point`**.

The incidence is $I(L,P)\iff h(L,P)=0$ and $P\in\mathcal H$.
Count by `i` to retain each line; count by `point` to retain each curve point.
The domain is explicitly 91 lines × 28 curve points. Later subrelations retain all
28 curve-point keys, including zero groups. Restricting to the stated universe
avoids computing and storing unused columns for the 63 non-curve points.

In [ ]:
def hermitian_incidence(points, p=3):
    n = points.count().scalar()
    curve = points.where(F.self_pair == 0).select().order_by(F.value)
    pairs = Collection.grid(n, curve.count().scalar(), values=1)
    left = tuple(points.bind(on=F.i, key=F.value, read=F[k]) for k in ('u','v','w'))
    right = tuple(curve.bind(on=F.j, key=F.index, read=F[k]) for k in ('u','v','w'))
    domain = pairs.annotate(pairing=hermitian_pair(left,right,p),
                           point=curve.bind(on=F.j,key=F.index,read=F.value))
    incidence = domain.where(F.pairing == 0)
    return incidence, incidence.count(by=F.i), incidence.count(by=F.point).order_by(F.key)

In [ ]:
incidence,line_counts,point_counts = hermitian_incidence(points,P)
incidence_workspace = Workspace({'points':points,'incidence':incidence,
                                  'line_counts':line_counts,'point_counts':point_counts})
assert not incidence_workspace.state.errors
ir = incidence_workspace.state.results
assert incidence.evaluate().cardinality == 280
assert np.count_nonzero(ir['line_counts'].values==1) == 28
assert np.count_nonzero(ir['line_counts'].values==4) == 63
curve_keys=np.flatnonzero(curve_snapshot.mask)
key_to_column={int(key):index for index,key in enumerate(curve_keys)}
assert ir['point_counts'].fields['key'].tolist() == curve_keys.tolist()
assert ir['point_counts'].values.tolist() == [10]*28
matrix = np.asarray(ir['incidence'].mask,dtype=int).reshape(91,28)
matrix_plot = style(go.Figure(go.Heatmap(z=matrix,x=list(map(str,curve_keys)),colorscale=[[0,'#152337'],[1,COLORS[0]]],
    showscale=False,hovertemplate='point %{x}<br>polar line %{y}<br>incidence %{z}<extra></extra>')),
    'One relation · 91 line poles × 28 curve points',height=590)
matrix_plot.update_xaxes(title='projective point key',type='category')
matrix_plot.update_yaxes(title='line-pole key i',autorange='reversed')
matrix_plot.show()
line_count_plot = profiles([ir['line_counts']],['Every line is tangent or secant'],
    keys=['i'],title='28 tangents touch once · 63 secants meet four times')
line_count_plot.show()
print('Incidences counted both ways:',sum(ir['line_counts'].values),sum(ir['point_counts'].values))

## 4 · A selected point becomes a lens argument

Select a pole and substitute its coordinates into $h(L,P)$. The resulting lens
highlights its polar line. The controls below inspect every captured pole, with
its line count, without changing the underlying construction.

For a curve point, the polar is its tangent and meets the curve only at that point.
For an external point, it is a secant containing four curve points. The matrix row,
highlighted points, and measured count describe the same incidence.

In [ ]:
line_frames=[]
for pole in range(91):
    selected=[bool(matrix[pole,key_to_column[key]]) if key in key_to_column else False for key in range(91)]
    marker_colors=[COLORS[1] if hit else (COLORS[0] if on_curve else '#2a3a50')
                   for hit,on_curve in zip(selected,curve_snapshot.mask)]
    line_frames.append(go.Frame(name=str(pole),data=[go.Scatter(marker=dict(color=marker_colors))],
        traces=[0],layout=go.Layout(title_text=f'Polar of point {pole} · {int(ir["line_counts"].values[pole])} curve points')))
polar_plot=go.Figure(curve_plot)
polar_plot.frames=line_frames
polar_plot.data[0].marker.color=line_frames[FOCUS].data[0].marker.color
polar_plot.update_layout(title_text=f'Polar of point {FOCUS} · 1 curve point',
    margin=dict(b=155),sliders=[dict(active=FOCUS,currentvalue=dict(prefix='Pole key: '),
        steps=[dict(label=str(k),method='animate',args=[[str(k)],
            dict(mode='immediate',frame=dict(duration=0,redraw=True),transition=dict(duration=0))]) for k in range(91)])])
polar_plot.show()

## 5 · Derive a partition before using it as a placement rule

Through the selected curve point, choose the **nine secants** and omit their common
point. Each remaining curve point belongs to exactly one chosen line. Three points
remain on each secant.

For an external pole, choose its **six secants** and add its polar line. These
seven blocks each contain four curve points and cover the curve once.

The curve-point pencil is completed by its tangent, whose curve intersection is
just the chosen point. This explicitly supplies the singleton block. Both complete
partitions now cover all 28 curve-point keys exactly once.

The function below declares four separate choices:

- **Grouping:** retain each point key in the partition incidence.
- **Coverage:** inspect counts and require exactly one contributor before reading its line key.
- **Ordering:** order the blocks by their keys, and their members by projective point key.
- **Placement:** bind the measured block order and within-block rank to named coordinates.

`coverage.unique(value=F.i)` preserves the weighted sum's contributors and fails if
any retained key has missing or multiple coverage. Its value zero can be a real
line key; it is never used as an implicit missing-value marker. The singleton's
separate display group is named `-1`, while its actual line field keeps its tangent
key. Member ranks use sorting and compact prefix evidence instead of dense pairs.

In [ ]:
def hermitian_partition(points, incidence, line_counts, focus, p=3, *, external=False):
    """Declare blocks, check unique coverage, and measure their order and ranks."""
    allowed = F.self_pair != 0 if external else F.self_pair == 0
    anchor = points.where((F.value == focus) & allowed).select()
    focus_coordinates = tuple(anchor.with_values(F[k]).scalar() for k in ('u','v','w'))
    through_focus = hermitian_pair((F.u,F.v,F.w),focus_coordinates,p) == 0
    chosen_family = (through_focus | (F.value == focus)) if external else through_focus
    secant_rule = chosen_family & (line_counts.bind(on=F.value,key=F.i) == p+1)
    secants = points.where(secant_rule)
    secant_flags = points.annotate(selected=choose(secant_rule,1,0))
    other_hits = incidence & incidence.universe.where(
        (F.point != focus) & (secant_flags.bind(on=F.i,key=F.value,read=F.selected) == 1))

    # A curve-point pencil gives nine triples and one singleton. Its tangent
    # supplies that singleton, so the complete partition covers all 28 keys once.
    partition = other_hits if external else other_hits | incidence.universe.where(
        (F.point == focus) & (F.i == focus))
    coverage = partition.group_by(F.point).coverage()
    cover_count = coverage.counts.order_by(F.key)
    unique_line = coverage.unique(value=F.i).order_by(F.key)

    curve = points.where(F.self_pair == 0).select()
    marked = curve.annotate(line=unique_line.bind(on=F.value))
    # -1 names the singleton's display group; F.line keeps its real tangent key.
    marked = marked.annotate(block=F.line if external else choose(F.value == focus,-1,F.line))
    blocks = marked.group_by(F.block)
    block_sizes = blocks.count()
    block_order = block_sizes.group_by().order_by(F.key).ranks()
    ranks = blocks.order_by(F.value).ranks(key=F.value).order_by(F.key)
    bundle = marked.annotate(
        bundle=block_order.bind(on=F.block) - (0 if external else 1),
        rank=ranks.bind(on=F.value))
    packed = bundle.arrange(x=3*F.bundle,y=choose(F.value == focus,1,F.rank))
    return dict(curve=curve,secants=secants,other_hits=other_hits,partition=partition,
                unique_line=unique_line,cover_count=cover_count,
                block_sizes=block_sizes,block_order=block_order,
                ranks=ranks,marked=bundle,packed=packed)

In [ ]:
pencil = hermitian_partition(points,incidence,line_counts,FOCUS,P)
spread = hermitian_partition(points,incidence,line_counts,EXTERNAL,P,external=True)
pencil_workspace,spread_workspace = Workspace(pencil),Workspace(spread)
assert not pencil_workspace.state.errors and not spread_workspace.state.errors
pr,sr = pencil_workspace.state.results,spread_workspace.state.results
assert pr['secants'].cardinality == 9 and sr['secants'].cardinality == 7
assert pr['other_hits'].cardinality == 27 and sr['other_hits'].cardinality == 28
assert pr['partition'].cardinality == sr['partition'].cardinality == 28
assert pr['cover_count'].values.tolist() == [1]*28
assert sr['cover_count'].values.tolist() == [1]*28
assert set(map(tuple,pr['packed'].positions)) == {(-3.,1.)} | {(3.*g,float(r)) for g in range(9) for r in range(3)}
assert set(map(tuple,sr['packed'].positions)) == {(3.*g,float(r)) for g in range(7) for r in range(4)}
assert pr['packed'].ids == sr['packed'].ids
print('One point + nine triples = seven quadruples =',len(pr['packed']))

motion_workspace = Workspace({'curve':pencil['curve']})
triples = motion_workspace.set('curve',pencil['packed'],motion=Motion.arc(height=2))
quadruples = motion_workspace.set('curve',spread['packed'],motion=Motion.arc(height=2))
motion_workspace.capture('Two incidence-defined partitions of the same 28 Hermitian points.')
unspread,unpencil = motion_workspace.undo(),motion_workspace.undo()
samples,labels=[],[]
for name,transition in [('one point + nine triples',triples),('seven quadruples',quadruples),
                        ('restore triples',unspread),('restore the curve',unpencil)]:
    for fraction in np.linspace(0,1,25):
        samples.append(transition.frame('curve',float(fraction)))
        labels.append(f'{name} · {fraction:.0%}')
colors_by_id={oid:('#edf2fa' if int(value)==FOCUS else GROUP_COLORS[int(group)%9])
             for oid,value,group in zip(pr['marked'].ids,pr['marked'].values,pr['marked'].fields['bundle'])}
motion_plot=replay(samples,labels,title='28 points · two measured partitions · reversible motion',
                   colors_by_id=colors_by_id)
motion_plot.show()
for forward,reverse in ((triples,unpencil),(quadruples,unspread)):
    np.testing.assert_array_equal(forward.frame('curve',.25).positions,reverse.frame('curve',.75).positions)
np.testing.assert_array_equal(samples[0].positions,samples[-1].positions)

Colors retain the original nine triples throughout this motion, so their mixing
inside the seven quadruples stays visible. White identifies the chosen curve point.
No point is deleted, duplicated, or silently matched by position.

## 6 · Remove the polar line: which points go missing?

The six secants through the external point alone miss four curve points. Those
are exactly the points of its polar line. This supplies a visible witness for why
the extra line is necessary, rather than accepting the equality of totals alone.

In [ ]:
incomplete = spread['other_hits'] & spread['other_hits'].universe.where(F.i != EXTERNAL)
incomplete_cover = incomplete.count(by=F.point).order_by(F.key)
uncovered = curve.select().where(incomplete_cover.bind(on=F.value,key=F.key)==0)
missing_snapshot=uncovered.evaluate()
missing_keys=set(map(int,missing_snapshot.source.values[missing_snapshot.mask]))
assert len(missing_keys)==4
assert missing_keys==set(curve_keys[np.flatnonzero(matrix[EXTERNAL])])
missing_plot=snapshot_figure(missing_snapshot,title='Omit the polar line · these four curve points are uncovered')
missing_plot.show()
print('Missing point keys:',sorted(missing_keys))

## Why these counts fit together

**Projective classes.** Nonzero scalar multiplication acts freely on nonzero
triples: if $\lambda v=v$ and a coordinate is nonzero, then $\lambda=1$.
Each class has eight representatives, giving $728/8=91$ classes.

**The curve has 28 points.** In $\mathbb F_9$, the norm fibers over $0,1,2$ have
sizes $1,4,4$. On the chart $(1,a,b)$, isotropy means $N(a)+N(b)=2$:
the possibilities $(0,2),(1,1),(2,0)$ give $4+16+4=24$ points.
The chart $(0,1,a)$ contributes four more with $N(a)=2$; $(0,0,1)$ contributes none.

**The line structure.** A Hermitian unital has one or $p+1$ points on each line,
and any pair of distinct curve points lies on one secant. These standard properties
give a $2$-$(28,4,1)$ design here. We verify every line and every pair independently
in the finite checks. The construction of a regular spread from an external pole
and its polar is described in [Dover, *A Search for Spreads of Hermitian Unitals*, §1](https://arxiv.org/pdf/1702.01297).

**The pencil.** Through one curve point, every other curve point lies on a unique
secant. Each secant contributes three other points, so $27/3=9$ triples.

**The external completion.** Its secants are disjoint on the curve because their
common pole is outside it. The four omitted points lie on its polar; adding that
block completes seven disjoint quadruples. Our coverage measurements check the
statement point by point, including the four witnesses in the incomplete case.

For Hermitian forms and the distinction between nondegeneracy and positivity, see
[Greaves, Iverson, Jasper, and Mixon, *Frames over finite fields*](https://arxiv.org/abs/2012.12977).
The motion explains how to inspect the finite construction; it does not establish a
new theorem for every field order, nor does it define a Euclidean curve.

In [ ]:
# An independent finite incidence invariant: any two curve points share one secant.
secant_rows=np.flatnonzero(ir['line_counts'].values==4)
blocks=matrix[secant_rows]
pair_counts=blocks.T @ blocks
assert np.all(pair_counts[np.triu_indices(28,1)]==1)
assert np.all(np.diag(pair_counts)==9)

INSPECT_POINT=next(int(k) for k,r in zip(pr['ranks'].fields['key'],pr['ranks'].values) if r>0)
point_index=pr['unique_line'].fields['key'].tolist().index(int(INSPECT_POINT))
line_key=int(pr['unique_line'].values[point_index])
contributors=pr['unique_line'].contributor_ids(int(INSPECT_POINT))
assert len(contributors)==1
measured_source=pr['other_hits'].source
source_index={oid:i for i,oid in enumerate(measured_source.ids)}
entry=source_index[contributors[0]]
assert int(measured_source.fields['i'][entry])==line_key
assert int(measured_source.fields['point'][entry])==INSPECT_POINT
representatives=qr['class_sizes'].contributor_ids(int(INSPECT_POINT))
assert len(representatives)==8
earlier=pr['ranks'].contributor_ids(INSPECT_POINT)
point_by_id=dict(zip(pr['curve'].ids,pr['curve'].values))
assert len(earlier)==int(pr['ranks'].values[point_index])
explanation={'point':int(INSPECT_POINT),'coordinates':point_labels[INSPECT_POINT],
    'assigned_secant':line_key,'coverage':1,'incidence_occurrence':contributors[0],
    'within_group_rank':int(pr['ranks'].values[point_index]),
    'projective_representatives':list(representatives),
    'rank_predecessor_keys':[int(point_by_id[oid]) for oid in earlier],
    'rank_source':pr['ranks'].metadata['universe']}

movie=write_mp4(samples,OUTPUT/'two-partitions-and-undo.mp4',labels=labels,
                title='The same 28 Hermitian points: triples, quadruples, undo',fps=24)
display(Video(str(movie),embed=True))
save_figures(OUTPUT,{'projective-quotient':quotient_plot,'hermitian-curve':curve_plot,
    'incidence-matrix':matrix_plot,'line-counts':line_count_plot,'polar-lens':polar_plot,
    'two-partitions':motion_plot,'missing-polar':missing_plot})
for name,ws in [('quotient',quotient_workspace),('quotient-motion',quotient_motion),
    ('incidence',incidence_workspace),('pencil',pencil_workspace),
    ('spread',spread_workspace),('motion',motion_workspace)]:
    # Compact whitespace only: retain every evaluated state, contributor, and path.
    payload=json.dumps(json.loads(ws.to_json()),separators=(',', ':'),allow_nan=False)
    (OUTPUT/f'{name}-workspace.json').write_text(payload)
    assert not Workspace.from_json(payload).state.errors
reopened=Workspace.from_json((OUTPUT/'motion-workspace.json').read_text())
replayed=reopened.redo()
np.testing.assert_array_equal(replayed.frame('curve',.25).positions,triples.frame('curve',.25).positions)
(OUTPUT/'point-explanation.json').write_text(json.dumps(explanation,indent=2))
(OUTPUT/'checks.json').write_text(json.dumps({'p':P,'focus':FOCUS,'external':EXTERNAL,
    'raw_vectors':728,'projective_points':91,'curve_points':28,'tangents':28,'secants':63,
    'pencil_groups':9,'spread_groups':7,'missing_polar_points':sorted(missing_keys),
    'motion_frames':len(samples),'quotient_frames':len(quotient_samples)},indent=2))
print('Saved seven offline views, an MP4, captured investigations, and the point-to-line explanation.')

This lesson asks for a future interface that can **select a point, use it as a
relation argument, inspect its polar, count line intersections, verify coverage,
and adopt the resulting correspondence as a layout**. A different representative
or chart should not change the projective point. A different pole changes the
mathematical question.

Possible next investigations include finite symmetries and Burnside counting,
syndrome classes in error-correcting codes, and measurement-preserving switches.
Their construction briefs and UI questions are preserved in
[future lessons](../docs/lessons/FUTURE_LESSONS.md).